# Avaliação da busca (Colab + GPU)

Compara a busca **densa**, **esparsa** e **híbrida**, cada uma **sem** e **com** reranking, nos conjuntos *título do TCC → orientador* e *IC (título; título + palavras-chave) → orientador*.
Métricas: nDCG@5, MRR, Recall@5 e Hit@1, com teste de Wilcoxon pareado contra a híbrida+rerank.

Antes: ative a GPU (*Ambiente de execução > Alterar o tipo*) e cadastre `QDRANT_URL` e `QDRANT_API_KEY` (e `GITHUB_TOKEN`, se o repositório for privado) nos *Secrets* do Colab.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Sem GPU: ative em Ambiente de execução > Alterar o tipo"

## 1. Código e dependências

In [ ]:
import os
from google.colab import userdata

REPO = "gavgms12/tcc_repository"
BRANCH = "buscas"
PASTA = "/content/tcc_repository"

try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

if os.path.exists(PASTA):
    !git -C {PASTA} fetch -q origin {BRANCH} && git -C {PASTA} checkout -q {BRANCH} && git -C {PASTA} reset -q --hard origin/{BRANCH}
else:
    !git clone -q --branch {BRANCH} {url} {PASTA}
!git -C {PASTA} log -1 --oneline

In [ ]:
# O torch com CUDA já vem no Colab; aqui entram só as libs da busca e da API.
!pip install -q -r {PASTA}/requirements-busca.txt fastapi uvicorn pandas python-dotenv
# Se o pip pedir para reiniciar a sessão: reinicie e rode a partir da célula de clone (é rápido na 2ª vez).

## 2. Cache do reranker no Google Drive

Reaproveita o mesmo cache da API: pares já calculados não rodam o modelo de novo.

In [ ]:
import shutil, sqlite3

USAR_DRIVE = True                                   # False: cache só desta sessão
CACHE_DRIVE = "/content/drive/MyDrive/tcc_cache/rerank_scores.sqlite"
CACHE_LOCAL = "/content/rerank_scores.sqlite"       # o SQLite trabalha no disco local (mais rápido e seguro)

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(os.path.dirname(CACHE_DRIVE), exist_ok=True)
    if os.path.exists(CACHE_DRIVE) and not os.path.exists(CACHE_LOCAL):
        shutil.copy(CACHE_DRIVE, CACHE_LOCAL)

os.environ["RERANK_CACHE_PATH"] = CACHE_LOCAL       # lido pelo config.py
if os.path.exists(CACHE_LOCAL):
    with sqlite3.connect(CACHE_LOCAL) as con:
        print("Notas no cache:", con.execute("SELECT COUNT(*) FROM scores").fetchone()[0])
else:
    print("Cache vazio: as notas serão calculadas na 1ª busca de cada tema/texto.")


def salvar_cache_no_drive():
    """Cópia consistente do SQLite para o Drive (backup do sqlite, seguro mesmo com a API rodando)."""
    if not USAR_DRIVE or not os.path.exists(CACHE_LOCAL):
        return
    temporario = CACHE_DRIVE + ".tmp"
    with sqlite3.connect(CACHE_LOCAL) as origem, sqlite3.connect(temporario) as destino:
        origem.backup(destino)
    os.replace(temporario, CACHE_DRIVE)

## 3. Configuração

In [ ]:
import sys, torch

os.environ["QDRANT_URL"] = userdata.get("QDRANT_URL")
os.environ["QDRANT_API_KEY"] = userdata.get("QDRANT_API_KEY")
os.chdir(PASTA)
if PASTA not in sys.path:
    sys.path.insert(0, PASTA)
print("GPU:", torch.cuda.is_available())

## 4. Rodar a avaliação

`--n 150` sorteia 150 TCCs (semente fixa); `--n 0` usa todos. Com GPU, conte alguns minutos por modo.

In [ ]:
N_TCCS = 150
CONJUNTO = "ambos"   # "tcc", "ic" ou "ambos" (IC: título e título + palavras-chave)
!cd {PASTA} && python -m recuperacao.sistema_de_busca.avaliacao.avaliar --conjunto {CONJUNTO} --n {N_TCCS}
salvar_cache_no_drive()

## 5. Resultados

In [ ]:
import glob, pandas as pd

pasta = sorted(glob.glob(f"{PASTA}/recuperacao/output/avaliacao_*"))[-1]
resumo = pd.read_csv(f"{pasta}/resumo.csv")
display(resumo.round(4))

# copia os arquivos para o Drive (ou baixe pela barra lateral)
if USAR_DRIVE:
    destino = "/content/drive/MyDrive/tcc_cache/" + os.path.basename(pasta)
    shutil.copytree(pasta, destino, dirs_exist_ok=True)
    print("Copiado para", destino)